# 02 — Breakfast Text Embeddings and Resumable Full Dataset Manifest

This notebook prepares the text side of the Breakfast proof-of-concept and validates the full Breakfast feature set.

It creates:

```text
Breakfast action mapping CSV
Breakfast action prompts
Breakfast split validation
Breakfast full feature/label manifest
CLIP ViT-B/16 text embeddings for action names
teacher-student PoC config
```

```text
Google Drive dataset
→ copy files to /content cache in small batches
→ np.load each cached .npy file
→ append successful rows to manifest
→ save progress after every batch
```


## 1. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Imports and central paths


In [ ]:
from pathlib import Path
import json
import random
import shutil
import subprocess
import sys
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

DRIVE_ROOT = Path('/content/drive/MyDrive')
DATA_ROOT = DRIVE_ROOT / 'mmf_tas_lab_data'
PROJECT_ROOT = DRIVE_ROOT / 'mmf_tas_lab_project'

BREAKFAST_ROOT = DATA_ROOT / 'zenodo_ms_tcn_data' / 'breakfast'
DRIVE_OUT_ROOT = DATA_ROOT / 'text_assisted_tas' / 'breakfast'
LOCAL_OUT_ROOT = Path('/content/text_assisted_tas_temp/breakfast')

# Local cache used for full manifest scan.
LOCAL_BREAKFAST_CACHE = Path('/content/breakfast_full_scan_cache')
LOCAL_BREAKFAST_FEATURE_DIR = LOCAL_BREAKFAST_CACHE / 'features'
LOCAL_BREAKFAST_GT_DIR = LOCAL_BREAKFAST_CACHE / 'groundTruth'

LOCAL_METADATA_DIR = LOCAL_OUT_ROOT / 'metadata'
LOCAL_TEXT_EMBEDDING_DIR = LOCAL_OUT_ROOT / 'text_embeddings'
LOCAL_CONFIG_DIR = LOCAL_OUT_ROOT / 'configs'
LOCAL_RESULTS_DIR = LOCAL_OUT_ROOT / 'results'

DRIVE_METADATA_DIR = DRIVE_OUT_ROOT / 'metadata'
DRIVE_TEXT_EMBEDDING_DIR = DRIVE_OUT_ROOT / 'text_embeddings'
DRIVE_CONFIG_DIR = DRIVE_OUT_ROOT / 'configs'
DRIVE_RESULTS_DIR = DRIVE_OUT_ROOT / 'results'

for path in [
    LOCAL_METADATA_DIR, LOCAL_TEXT_EMBEDDING_DIR, LOCAL_CONFIG_DIR, LOCAL_RESULTS_DIR,
    DRIVE_METADATA_DIR, DRIVE_TEXT_EMBEDDING_DIR, DRIVE_CONFIG_DIR, DRIVE_RESULTS_DIR,
    LOCAL_BREAKFAST_FEATURE_DIR, LOCAL_BREAKFAST_GT_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print('BREAKFAST_ROOT:', BREAKFAST_ROOT)
print('LOCAL_OUT_ROOT:', LOCAL_OUT_ROOT)
print('DRIVE_OUT_ROOT:', DRIVE_OUT_ROOT)
print('LOCAL_BREAKFAST_CACHE:', LOCAL_BREAKFAST_CACHE)


## 3. Configuration


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

RUN_CLIP_TEXT_EMBEDDING_EXTRACTION = True
DEBUG_NUM_VIDEOS_PER_SPLIT = 3

# Full manifest scan options
FULL_SCAN_ALL_FEATURE_FILES = True

# Required for stable Colab behavior
COPY_DATA_TO_LOCAL_FOR_FULL_SCAN = True

FULL_SCAN_BATCH_SIZE = 25
MAX_COPY_RETRIES = 2

FAIL_ON_REAL_FILE_ERRORS = True

# Some TAS datasets can have minor feature/label length differences
FAIL_ON_TIMESTEP_MISMATCH = False

print('SEED:', SEED)
print('RUN_CLIP_TEXT_EMBEDDING_EXTRACTION:', RUN_CLIP_TEXT_EMBEDDING_EXTRACTION)
print('DEBUG_NUM_VIDEOS_PER_SPLIT:', DEBUG_NUM_VIDEOS_PER_SPLIT)
print('FULL_SCAN_ALL_FEATURE_FILES:', FULL_SCAN_ALL_FEATURE_FILES)
print('COPY_DATA_TO_LOCAL_FOR_FULL_SCAN:', COPY_DATA_TO_LOCAL_FOR_FULL_SCAN)
print('FULL_SCAN_BATCH_SIZE:', FULL_SCAN_BATCH_SIZE)
print('MAX_COPY_RETRIES:', MAX_COPY_RETRIES)
print('FAIL_ON_REAL_FILE_ERRORS:', FAIL_ON_REAL_FILE_ERRORS)
print('FAIL_ON_TIMESTEP_MISMATCH:', FAIL_ON_TIMESTEP_MISMATCH)


## 4. Helper functions


In [ ]:
def read_lines(path: Path):
    return path.read_text().splitlines()


def require_exists(path: Path, name: str):
    if not path.exists():
        raise FileNotFoundError(f'{name} not found: {path}')
    return path


def save_csv_local_then_drive(df: pd.DataFrame, local_path: Path, drive_path: Path | None = None):
    local_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(local_path, index=False)
    print('Saved local:', local_path)

    if drive_path is not None:
        try:
            drive_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(local_path, drive_path)
            print('Copied to Drive:', drive_path)
        except Exception as e:
            print('WARNING: Could not copy to Drive.')
            print('Local file is still safe:', local_path)
            print('Error:', repr(e))


def load_mapping(mapping_path: Path):
    rows = []
    idx_to_label = {}
    label_to_idx = {}

    for line in read_lines(mapping_path):
        line = line.strip()
        if not line:
            continue
        idx, label = line.split(maxsplit=1)
        idx = int(idx)
        rows.append({'class_index': idx, 'action_label': label})
        idx_to_label[idx] = label
        label_to_idx[label] = idx

    df_mapping = pd.DataFrame(rows).sort_values('class_index').reset_index(drop=True)
    return df_mapping, idx_to_label, label_to_idx


def load_split(split_path: Path):
    video_ids = []
    for line in read_lines(split_path):
        line = line.strip()
        if not line:
            continue
        video_ids.append(Path(line).stem)
    return video_ids


def inspect_feature(feature_path: Path):
    arr = np.load(feature_path, mmap_mode='r')
    if arr.ndim != 2:
        raise ValueError(f'Expected 2D feature array [D, T], got shape={arr.shape}')

    return {
        'shape': tuple(arr.shape),
        'dtype': str(arr.dtype),
        'feature_dim': int(arr.shape[0]),
        'timesteps': int(arr.shape[1]),
    }


def safe_csv_num_rows(path: Path) -> int:
    if not path.exists():
        return -1
    if path.stat().st_size == 0:
        return 0
    try:
        return len(pd.read_csv(path))
    except pd.errors.EmptyDataError:
        return 0


def copy_if_needed(src: Path, dst: Path):
    """Copy src to dst if dst is missing or has a different file size."""
    dst.parent.mkdir(parents=True, exist_ok=True)

    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        return False

    shutil.copy2(src, dst)
    return True


def append_csv_safely(df: pd.DataFrame, path: Path):
    """Append df rows to CSV. Creates file with header if missing/empty."""
    path.parent.mkdir(parents=True, exist_ok=True)

    write_header = (not path.exists()) or path.stat().st_size == 0
    df.to_csv(path, mode='a', header=write_header, index=False)


def try_copy_with_retries(src: Path, dst: Path, max_retries: int = 2):
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            changed = copy_if_needed(src, dst)
            return True, changed, None
        except Exception as e:
            last_error = e
    return False, False, last_error


## 5. Validate Breakfast files


In [ ]:
def list_split_files(splits_dir: Path):
    """
    Return all Breakfast split bundle files.

    Expected files look like:
    train.split1.bundle
    test.split1.bundle
    ...
    """
    split_files = sorted(list(splits_dir.glob("*.bundle")))

    if len(split_files) == 0:
        raise FileNotFoundError(f"No .bundle split files found in: {splits_dir}")

    return split_files

required_paths = {
    'Breakfast root': BREAKFAST_ROOT,
    'features': BREAKFAST_ROOT / 'features',
    'groundTruth': BREAKFAST_ROOT / 'groundTruth',
    'mapping.txt': BREAKFAST_ROOT / 'mapping.txt',
    'splits': BREAKFAST_ROOT / 'splits',
}

for description, path in required_paths.items():
    require_exists(path, description)

feature_files = sorted((BREAKFAST_ROOT / 'features').glob('*.npy'))
gt_files = sorted((BREAKFAST_ROOT / 'groundTruth').glob('*.txt'))
split_files = list_split_files(BREAKFAST_ROOT / 'splits')

print('\nCounts:')
print('Feature files:', len(feature_files))
print('Ground-truth files:', len(gt_files))
print('Split files:', len(split_files))

assert len(feature_files) == 1712, f'Expected 1712 feature files, found {len(feature_files)}'
assert len(gt_files) == 1712, f'Expected 1712 groundTruth files, found {len(gt_files)}'


## 6. Load action mapping and create text prompts


In [ ]:
df_mapping, idx_to_label, label_to_idx = load_mapping(BREAKFAST_ROOT / "mapping.txt")

print("Number of classes:", len(df_mapping))
display(df_mapping.head(20))

save_csv_local_then_drive(
    df_mapping,
    LOCAL_METADATA_DIR / "breakfast_action_mapping.csv",
    DRIVE_METADATA_DIR / "breakfast_action_mapping.csv",
)


def label_to_readable_text(label: str):
    return label.replace("_", " ").strip()


df_prompts = df_mapping.copy()

# Make column names robust, in case mapping uses action_label instead of label
if "label" in df_prompts.columns:
    label_col = "label"
elif "action_label" in df_prompts.columns:
    label_col = "action_label"
else:
    raise KeyError(f"Could not find label column. Available columns: {list(df_prompts.columns)}")

df_prompts["action_text"] = df_prompts[label_col].apply(label_to_readable_text)
df_prompts["prompt"] = df_prompts["action_text"].apply(
    lambda x: f"a video of the action: {x}"
)

display(df_prompts.head(20))

save_csv_local_then_drive(
    df_prompts,
    LOCAL_METADATA_DIR / "breakfast_action_text_prompts.csv",
    DRIVE_METADATA_DIR / "breakfast_action_text_prompts.csv",
)

## 7. Validate splits


In [ ]:
split_summary_rows = []

for split_file in split_files:
    video_ids = load_split(split_file)
    missing_features = []
    missing_labels = []

    for video_id in video_ids:
        if not (BREAKFAST_ROOT / 'features' / f'{video_id}.npy').exists():
            missing_features.append(video_id)
        if not (BREAKFAST_ROOT / 'groundTruth' / f'{video_id}.txt').exists():
            missing_labels.append(video_id)

    split_summary_rows.append({
        'split_file': split_file.name,
        'num_videos': len(video_ids),
        'missing_features': len(missing_features),
        'missing_labels': len(missing_labels),
        'missing_feature_examples': missing_features[:3],
        'missing_label_examples': missing_labels[:3],
    })

df_splits = pd.DataFrame(split_summary_rows)
display(df_splits)

save_csv_local_then_drive(
    df_splits,
    LOCAL_METADATA_DIR / 'breakfast_split_validation.csv',
    DRIVE_METADATA_DIR / 'breakfast_split_validation.csv',
)

if (df_splits['missing_features'].sum() + df_splits['missing_labels'].sum()) > 0:
    raise RuntimeError('Some split entries are missing features or labels.')


## 8. Inspect sample feature/label alignment


In [ ]:
sample_rows = []

for split_file in split_files[:2]:
    video_ids = load_split(split_file)[:DEBUG_NUM_VIDEOS_PER_SPLIT]

    for video_id in video_ids:
        feature_path = BREAKFAST_ROOT / 'features' / f'{video_id}.npy'
        gt_path = BREAKFAST_ROOT / 'groundTruth' / f'{video_id}.txt'

        feature_info = inspect_feature(feature_path)
        labels = read_lines(gt_path)

        sample_rows.append({
            'split_file': split_file.name,
            'video_id': video_id,
            'feature_shape': str(feature_info['shape']),
            'feature_dim': feature_info['feature_dim'],
            'feature_timesteps': feature_info['timesteps'],
            'label_timesteps': len(labels),
            'timestep_diff': feature_info['timesteps'] - len(labels),
        })

df_sample_align = pd.DataFrame(sample_rows)
display(df_sample_align)

save_csv_local_then_drive(
    df_sample_align,
    LOCAL_METADATA_DIR / 'breakfast_sample_feature_label_alignment.csv',
    DRIVE_METADATA_DIR / 'breakfast_sample_feature_label_alignment.csv',
)


## 9. Build resumable full Breakfast feature/label manifest


In [ ]:
if not FULL_SCAN_ALL_FEATURE_FILES:
    raise RuntimeError('This notebook version is intended to run FULL_SCAN_ALL_FEATURE_FILES=True.')

feature_ids = {p.stem for p in feature_files}
gt_ids = {p.stem for p in gt_files}
common_ids = sorted(feature_ids & gt_ids)

missing_feature_ids = sorted(gt_ids - feature_ids)
missing_gt_ids = sorted(feature_ids - gt_ids)

print('Feature ids:', len(feature_ids))
print('Ground-truth ids:', len(gt_ids))
print('Common feature/label ids:', len(common_ids))
print('Missing feature ids:', len(missing_feature_ids))
print('Missing ground-truth ids:', len(missing_gt_ids))

manifest_columns = [
    'video_id',
    'feature_path',
    'ground_truth_path',
    'scan_feature_path',
    'scan_ground_truth_path',
    'feature_shape',
    'feature_dtype',
    'feature_dim',
    'feature_timesteps',
    'label_timesteps',
    'timestep_diff',
    'timestep_match',
    'manifest_mode',
]

bad_columns = [
    'video_id',
    'feature_path',
    'ground_truth_path',
    'error_stage',
    'error_type',
    'error_message',
]

local_manifest_path = LOCAL_METADATA_DIR / 'breakfast_dataset_manifest.csv'
local_bad_features_path = LOCAL_METADATA_DIR / 'breakfast_bad_feature_files.csv'
local_scan_state_path = LOCAL_METADATA_DIR / 'breakfast_full_scan_state.json'

drive_manifest_path = DRIVE_METADATA_DIR / 'breakfast_dataset_manifest.csv'
drive_bad_features_path = DRIVE_METADATA_DIR / 'breakfast_bad_feature_files.csv'
drive_scan_state_path = DRIVE_METADATA_DIR / 'breakfast_full_scan_state.json'

# Resume from existing local manifest if it exists
if local_manifest_path.exists() and local_manifest_path.stat().st_size > 0:
    try:
        existing_manifest = pd.read_csv(local_manifest_path)
        processed_ids = set(existing_manifest['video_id'].astype(str).tolist())
        print('Existing local manifest found:', local_manifest_path)
        print('Already processed videos:', len(processed_ids))
    except Exception as e:
        print('WARNING: Could not read existing manifest. Starting a fresh manifest.')
        print('Error:', repr(e))
        processed_ids = set()
        local_manifest_path.unlink(missing_ok=True)
else:
    processed_ids = set()

if local_bad_features_path.exists():
    local_bad_features_path.unlink()

remaining_ids = [vid for vid in common_ids if vid not in processed_ids]
print('Remaining videos to full-scan:', len(remaining_ids))

all_new_bad_rows = []
new_manifest_rows_total = 0
drive_disconnected = False

for batch_start in tqdm(range(0, len(remaining_ids), FULL_SCAN_BATCH_SIZE), desc='Full-scan batches'):
    batch_ids = remaining_ids[batch_start:batch_start + FULL_SCAN_BATCH_SIZE]

    batch_manifest_rows = []
    batch_bad_rows = []

    copied_features = 0
    copied_labels = 0
    copied_ok_ids = []

    for video_id in batch_ids:
        drive_feature_path = BREAKFAST_ROOT / 'features' / f'{video_id}.npy'
        drive_gt_path = BREAKFAST_ROOT / 'groundTruth' / f'{video_id}.txt'

        local_feature_path = LOCAL_BREAKFAST_FEATURE_DIR / f'{video_id}.npy'
        local_gt_path = LOCAL_BREAKFAST_GT_DIR / f'{video_id}.txt'

        ok_feature, changed_feature, err_feature = try_copy_with_retries(
            drive_feature_path,
            local_feature_path,
            max_retries=MAX_COPY_RETRIES,
        )

        ok_gt, changed_gt, err_gt = try_copy_with_retries(
            drive_gt_path,
            local_gt_path,
            max_retries=MAX_COPY_RETRIES,
        )

        if ok_feature and ok_gt:
            copied_ok_ids.append(video_id)
            copied_features += int(changed_feature)
            copied_labels += int(changed_gt)
        else:
            err = err_feature if err_feature is not None else err_gt
            batch_bad_rows.append({
                'video_id': video_id,
                'feature_path': str(drive_feature_path),
                'ground_truth_path': str(drive_gt_path),
                'error_stage': 'copy_to_local_cache',
                'error_type': type(err).__name__ if err is not None else 'UnknownCopyError',
                'error_message': repr(err),
            })

            if err is not None and ('Transport endpoint is not connected' in repr(err)):
                drive_disconnected = True
                break

    if drive_disconnected:
        print('\nGoogle Drive mount appears to be disconnected.')
        print('Stopping cleanly. Restart runtime, remount Drive, and rerun this notebook.')
        print('Already processed videos are saved in:', local_manifest_path)
        break

    # Full np.load scan only for successfully copied files
    for video_id in copied_ok_ids:
        original_feature_path = BREAKFAST_ROOT / 'features' / f'{video_id}.npy'
        original_gt_path = BREAKFAST_ROOT / 'groundTruth' / f'{video_id}.txt'

        scan_feature_path = LOCAL_BREAKFAST_FEATURE_DIR / f'{video_id}.npy'
        scan_gt_path = LOCAL_BREAKFAST_GT_DIR / f'{video_id}.txt'

        try:
            feature_info = inspect_feature(scan_feature_path)
            labels = read_lines(scan_gt_path)

            feature_timesteps = int(feature_info['timesteps'])
            label_timesteps = int(len(labels))
            timestep_diff = feature_timesteps - label_timesteps

            batch_manifest_rows.append({
                'video_id': video_id,
                'feature_path': str(original_feature_path),
                'ground_truth_path': str(original_gt_path),
                'scan_feature_path': str(scan_feature_path),
                'scan_ground_truth_path': str(scan_gt_path),
                'feature_shape': str(feature_info['shape']),
                'feature_dtype': feature_info['dtype'],
                'feature_dim': int(feature_info['feature_dim']),
                'feature_timesteps': feature_timesteps,
                'label_timesteps': label_timesteps,
                'timestep_diff': timestep_diff,
                'timestep_match': timestep_diff == 0,
                'manifest_mode': 'full_npy_scan_local_cache_resumable',
            })

        except Exception as e:
            batch_bad_rows.append({
                'video_id': video_id,
                'feature_path': str(original_feature_path),
                'ground_truth_path': str(original_gt_path),
                'error_stage': 'full_scan_np_load_or_label_read',
                'error_type': type(e).__name__,
                'error_message': repr(e),
            })

    # Append only successful new rows
    if batch_manifest_rows:
        df_batch_manifest = pd.DataFrame(batch_manifest_rows, columns=manifest_columns)
        append_csv_safely(df_batch_manifest, local_manifest_path)
        new_manifest_rows_total += len(df_batch_manifest)

    if batch_bad_rows:
        df_batch_bad = pd.DataFrame(batch_bad_rows, columns=bad_columns)
        append_csv_safely(df_batch_bad, local_bad_features_path)
        all_new_bad_rows.extend(batch_bad_rows)

    # Save state after every batch
    state = {
        'total_common_ids': len(common_ids),
        'already_processed_before_run': len(processed_ids),
        'new_manifest_rows_this_run': new_manifest_rows_total,
        'last_batch_start': batch_start,
        'drive_disconnected': drive_disconnected,
        'local_manifest_path': str(local_manifest_path),
        'local_bad_features_path': str(local_bad_features_path),
    }

    with local_scan_state_path.open('w') as f:
        json.dump(state, f, indent=2)

if local_manifest_path.exists() and local_manifest_path.stat().st_size > 0:
    df_manifest = pd.read_csv(local_manifest_path)
else:
    df_manifest = pd.DataFrame(columns=manifest_columns)

if local_bad_features_path.exists() and local_bad_features_path.stat().st_size > 0:
    df_bad_features = pd.read_csv(local_bad_features_path)
else:
    df_bad_features = pd.DataFrame(columns=bad_columns)

print('\nCurrent full-scan status:')
print('Manifest rows:', len(df_manifest))
print('Bad rows from current run:', len(df_bad_features))
print('Expected rows:', len(common_ids))
print('Remaining rows:', len(common_ids) - len(df_manifest))

if len(df_manifest) > 0:
    print('\nFeature dimension counts:')
    display(df_manifest['feature_dim'].value_counts().sort_index().rename_axis('feature_dim').reset_index(name='count'))

    print('\nTimestep match counts:')
    display(df_manifest['timestep_match'].value_counts(dropna=False).rename_axis('timestep_match').reset_index(name='count'))

    mismatch_rows = df_manifest[df_manifest['timestep_match'] == False].copy()
    print('Timestep mismatches:', len(mismatch_rows))

    display(df_manifest.head())

    if len(mismatch_rows) > 0:
        print('\nFirst timestep mismatches:')
        display(mismatch_rows.head(20))

if len(df_bad_features) > 0:
    print('\nBad rows from the current run:')
    display(df_bad_features.head(20))

save_csv_local_then_drive(df_manifest, local_manifest_path, drive_manifest_path)
save_csv_local_then_drive(df_bad_features, local_bad_features_path, drive_bad_features_path)

try:
    shutil.copy2(local_scan_state_path, drive_scan_state_path)
    print('Copied scan state to Drive:', drive_scan_state_path)
except Exception as e:
    print('WARNING: Could not copy scan state to Drive.')
    print('Local scan state is safe:', local_scan_state_path)
    print('Error:', repr(e))

if drive_disconnected:
    raise RuntimeError(
        'Google Drive disconnected during copy. This is not a data corruption error. '
        'Restart runtime, remount Drive, and rerun this notebook; it will continue from the saved manifest.'
    )

if len(df_manifest) < len(common_ids):
    raise RuntimeError(
        f'Full scan is incomplete: {len(df_manifest)}/{len(common_ids)} videos processed. '
        'Rerun the notebook to continue.'
    )

if len(df_bad_features) > 0 and FAIL_ON_REAL_FILE_ERRORS:
    raise RuntimeError('Some files failed real processing. See df_bad_features above.')

if len(df_manifest) > 0:
    mismatch_count = int((df_manifest['timestep_match'] == False).sum())
    if mismatch_count > 0 and FAIL_ON_TIMESTEP_MISMATCH:
        raise RuntimeError(f'Found {mismatch_count} feature/label timestep mismatches.')

print('\nFull Breakfast manifest scan completed successfully.')


## 10. Generate CLIP text embeddings


In [ ]:
if RUN_CLIP_TEXT_EMBEDDING_EXTRACTION:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy', 'regex', 'tqdm'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/openai/CLIP.git'], check=True)
    print('CLIP dependencies installed.')
else:
    print('Skipping CLIP installation.')


In [ ]:
if RUN_CLIP_TEXT_EMBEDDING_EXTRACTION:
    import torch
    import clip

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('Device:', device)

    clip_model_name = 'ViT-B/16'
    model, _ = clip.load(clip_model_name, device=device)
    model.eval()

    prompts = df_prompts['prompt'].tolist()
    tokens = clip.tokenize(prompts, truncate=True).to(device)

    with torch.no_grad():
        text_embeddings = model.encode_text(tokens).float()
        text_embeddings = text_embeddings / text_embeddings.norm(dim=-1, keepdim=True)

    text_embeddings_np = text_embeddings.cpu().numpy().astype(np.float32)

    local_embedding_path = LOCAL_TEXT_EMBEDDING_DIR / 'breakfast_clip_vitb16_text_embeddings.npy'
    local_metadata_path = LOCAL_TEXT_EMBEDDING_DIR / 'breakfast_clip_vitb16_text_embedding_metadata.csv'
    local_config_path = LOCAL_TEXT_EMBEDDING_DIR / 'breakfast_clip_vitb16_text_embedding_config.json'

    np.save(local_embedding_path, text_embeddings_np)
    df_prompts.to_csv(local_metadata_path, index=False)

    text_embedding_config = {
        'dataset': 'breakfast',
        'encoder': 'OpenAI CLIP',
        'model': clip_model_name,
        'num_classes': int(text_embeddings_np.shape[0]),
        'embedding_dim': int(text_embeddings_np.shape[1]),
        'prompt_template': 'a video of the action: <action name>',
        'embedding_file': str(DRIVE_TEXT_EMBEDDING_DIR / local_embedding_path.name),
        'metadata_file': str(DRIVE_TEXT_EMBEDDING_DIR / local_metadata_path.name),
    }

    with local_config_path.open('w') as f:
        json.dump(text_embedding_config, f, indent=2)

    print('Saved local text embeddings:', local_embedding_path)
    print('Text embedding shape:', text_embeddings_np.shape)

    try:
        for src_path in [local_embedding_path, local_metadata_path, local_config_path]:
            shutil.copy2(src_path, DRIVE_TEXT_EMBEDDING_DIR / src_path.name)
            print('Copied to Drive:', DRIVE_TEXT_EMBEDDING_DIR / src_path.name)
    except Exception as e:
        print('WARNING: Could not copy text embeddings to Drive.')
        print('Local text files are safe under:', LOCAL_TEXT_EMBEDDING_DIR)
        print('Error:', repr(e))
else:
    print('Skipping CLIP text embedding extraction.')


## 11. Validate text embeddings and save PoC config


In [ ]:
local_embedding_path = LOCAL_TEXT_EMBEDDING_DIR / 'breakfast_clip_vitb16_text_embeddings.npy'
local_metadata_path = LOCAL_TEXT_EMBEDDING_DIR / 'breakfast_clip_vitb16_text_embedding_metadata.csv'

if not local_embedding_path.exists():
    drive_embedding_path = DRIVE_TEXT_EMBEDDING_DIR / local_embedding_path.name
    drive_metadata_path = DRIVE_TEXT_EMBEDDING_DIR / local_metadata_path.name
    require_exists(drive_embedding_path, 'Drive text embedding file')
    require_exists(drive_metadata_path, 'Drive text embedding metadata')
    shutil.copy2(drive_embedding_path, local_embedding_path)
    shutil.copy2(drive_metadata_path, local_metadata_path)

loaded_embeddings = np.load(local_embedding_path)
loaded_metadata = pd.read_csv(local_metadata_path)

print('Loaded text embedding shape:', loaded_embeddings.shape)
print('Loaded metadata rows:', len(loaded_metadata))
print('Expected classes:', len(df_mapping))

assert loaded_embeddings.shape[0] == len(df_mapping)
assert len(loaded_metadata) == len(df_mapping)

display(loaded_metadata.head())

expected_feature_dim = int(df_manifest['feature_dim'].mode()[0])

poc_config = {
    'project': 'text_assisted_temporal_action_segmentation',
    'dataset': 'breakfast',
    'baseline_model': 'MS-TCN-style temporal convolutional network',
    'visual_features': {
        'source': 'Zenodo MS-TCN Breakfast I3D features',
        'feature_dir': str(BREAKFAST_ROOT / 'features'),
        'shape_convention': '[D, T]',
        'expected_feature_dim': expected_feature_dim,
    },
    'text_features': {
        'encoder': 'CLIP ViT-B/16',
        'source': 'Breakfast action class names',
        'prompt_template': 'a video of the action: <action name>',
        'embedding_file': str(DRIVE_TEXT_EMBEDDING_DIR / 'breakfast_clip_vitb16_text_embeddings.npy'),
        'metadata_file': str(DRIVE_TEXT_EMBEDDING_DIR / 'breakfast_clip_vitb16_text_embedding_metadata.csv'),
    },
    'teacher_student_setup': {
        'teacher': 'text-aware teacher using visual features and CLIP action prototypes',
        'student': 'video-only student',
        'student_inference_requires_text': False,
        'losses': ['cross_entropy', 'knowledge_distillation'],
    },
    'evaluation_plan': ['Acc', 'Edit', 'F1@10', 'F1@25', 'F1@50'],
}

local_config_path = LOCAL_CONFIG_DIR / 'breakfast_mstcn_teacher_student_poc_config.json'
drive_config_path = DRIVE_CONFIG_DIR / 'breakfast_mstcn_teacher_student_poc_config.json'

with local_config_path.open('w') as f:
    json.dump(poc_config, f, indent=2)

try:
    shutil.copy2(local_config_path, drive_config_path)
except Exception as e:
    print('WARNING: Could not copy PoC config to Drive:', repr(e))

print(json.dumps(poc_config, indent=2))


## 12. Final output check


In [ ]:
local_expected = [
    LOCAL_METADATA_DIR / 'breakfast_action_mapping.csv',
    LOCAL_METADATA_DIR / 'breakfast_action_text_prompts.csv',
    LOCAL_METADATA_DIR / 'breakfast_split_validation.csv',
    LOCAL_METADATA_DIR / 'breakfast_sample_feature_label_alignment.csv',
    LOCAL_METADATA_DIR / 'breakfast_dataset_manifest.csv',
    LOCAL_METADATA_DIR / 'breakfast_bad_feature_files.csv',
    LOCAL_TEXT_EMBEDDING_DIR / 'breakfast_clip_vitb16_text_embeddings.npy',
    LOCAL_TEXT_EMBEDDING_DIR / 'breakfast_clip_vitb16_text_embedding_metadata.csv',
    LOCAL_TEXT_EMBEDDING_DIR / 'breakfast_clip_vitb16_text_embedding_config.json',
    LOCAL_CONFIG_DIR / 'breakfast_mstcn_teacher_student_poc_config.json',
]

print('LOCAL OUTPUT CHECK')
for p in local_expected:
    print(f'{p.exists()}  {p}')

print('\nDRIVE OUTPUT CHECK')
drive_expected = [
    DRIVE_METADATA_DIR / 'breakfast_dataset_manifest.csv',
    DRIVE_TEXT_EMBEDDING_DIR / 'breakfast_clip_vitb16_text_embeddings.npy',
    DRIVE_CONFIG_DIR / 'breakfast_mstcn_teacher_student_poc_config.json',
]
for p in drive_expected:
    print(f'{p.exists()}  {p}')

print('\nSummary values:')
print('Feature files:', len(feature_files))
print('Ground-truth files:', len(gt_files))
print('Classes:', len(df_mapping))
print('Manifest rows:', safe_csv_num_rows(LOCAL_METADATA_DIR / 'breakfast_dataset_manifest.csv'))
print('Bad feature rows:', safe_csv_num_rows(LOCAL_METADATA_DIR / 'breakfast_bad_feature_files.csv'))
print('Text embedding shape:', np.load(LOCAL_TEXT_EMBEDDING_DIR / 'breakfast_clip_vitb16_text_embeddings.npy').shape)
print('\n02_text_embeddings_breakfast completed.')
